In [1]:
# [1] Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# [2] Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


# temperature=0 so score changes across runs come from the prompt, not sampling noise
def chat(messages, system=None, temperature=0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# [3] Function to generate a new dataset
import json
import re


def safe_json_loads(text):
    """Parse JSON from model output, repairing stray backslashes
    (e.g. unescaped regex like \\d or \\. inside a JSON string)."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        fixed = re.sub(r'\\(?!["\\/bfnrtu])', r"\\\\", text)
        return json.loads(fixed)


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return safe_json_loads(text)

In [4]:
# [4] Generate the dataset  --  DO NOT RUN DURING THE DEMO (changes dataset-aws.json)
dataset = generate_dataset()
with open("dataset-aws.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
# [5] Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
If you quote a regex or any text containing backslashes, escape them so the result is valid JSON (use \\\\d, not \\d).
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return safe_json_loads(eval_text)

In [ ]:
# [6] Prompt versions  --  EDIT + RUN every demo iteration
#
# Each prompt version is one step of the demo. Set PROMPT_VERSION, re-run the eval
# cell, read the weaknesses in the report, then write the next version.
# Every version returns (prompt, assistant_prefill).

PROMPT_VERSION = "v2"


def prompt_v1(test_case):
    """Baseline - a reasonable first attempt. States the required format, but
    says nothing about what that artifact should actually look like, never shows
    Claude the criteria it will be graded on, and asks for validation/error
    handling on tasks that don't need it."""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

Required output format: {test_case["format"]}

* Respond only with the {test_case["format"]}, no commentary or explanation
* Handle input validation and errors
"""
    return prompt, "```code"


def prompt_v2(test_case):
    """+ spells out the exact shape each format must take
    + shows the solver the criteria it will be graded against
    + drops the instruction that was causing over-engineering
    + prefills the fence with the requested format"""
    format = test_case["format"]
    prompt = f"""
Please solve the following task:

{test_case["task"]}

Required output format: {format}

Your solution must satisfy these criteria:
<criteria>
{test_case["solution_criteria"]}
</criteria>

* Respond with a single {format} artifact and nothing else
* json  -> one bare JSON object. Not Python that prints JSON
* python -> a single function definition. No example usage, no prints, no __main__
* regex -> the raw pattern only, on one line. No re.compile(...), no quotes, no delimiters
* Do not add any comments or commentary or explanation
* Solve exactly what is asked - no extra dependencies, no AWS API calls
"""
    return prompt, f"```{format}"


PROMPTS = {
    "v1": prompt_v1,
    "v2": prompt_v2,
}


def run_prompt(test_case):
    prompt, prefill = PROMPTS[PROMPT_VERSION](test_case)

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, prefill)
    output = chat(messages, stop_sequences=["```"])
    return output

In [7]:
# [7] Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [8]:
# [8] Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    weaknesses = model_grade.get("weaknesses", [])

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        # Kept separately so the report shows WHICH half of the score moved
        "model_score": model_score,
        "syntax_score": syntax_score,
        "reasoning": reasoning,
        "weaknesses": weaknesses,
    }

In [9]:
# [9] Run the eval across the whole dataset
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [10]:
# [10] Write the results to a Markdown file so they're readable
from datetime import datetime


def write_results_md(results, path=None, version=None):
    """Render eval results as a Markdown report: summary table + one section per test case."""
    version = version or PROMPT_VERSION
    path = path or f"prompt-eval-results-{version}.md"
    average_score = mean([result["score"] for result in results])

    lines = [
        f"# Prompt Eval Results - {version}",
        "",
        f"- **Model:** `{model}`",
        f"- **Prompt version:** `{version}`",
        f"- **Generated:** {datetime.now():%Y-%m-%d %H:%M}",
        f"- **Test cases:** {len(results)}",
        f"- **Average score:** {average_score:.2f} / 10",
        "",
        "| # | Format | Score | Syntax | Model | Task |",
        "| --- | --- | --- | --- | --- | --- |",
    ]

    for index, result in enumerate(results, start=1):
        test_case = result["test_case"]
        # Escape pipes so a task description can't break the table
        task = test_case["task"].replace("|", "\\|")
        format = test_case.get("format", "-")
        lines.append(
            f"| {index} | {format} | {result['score']} | "
            f"{result.get('syntax_score', '-')} | {result.get('model_score', '-')} | {task} |"
        )

    for index, result in enumerate(results, start=1):
        test_case = result["test_case"]
        lines += [
            "",
            f"## {index}. {test_case['task']}",
            "",
            f"- **Format:** {test_case.get('format', '-')}",
            f"- **Score:** {result['score']} / 10"
            f" (syntax {result.get('syntax_score', '-')}, model {result.get('model_score', '-')})",
        ]

        if "solution_criteria" in test_case:
            lines.append(f"- **Criteria:** {test_case['solution_criteria']}")

        weaknesses = result.get("weaknesses") or []
        if weaknesses:
            lines += ["", "**Weaknesses**", ""]
            lines += [f"- {weakness}" for weakness in weaknesses]

        lines += [
            "",
            "**Reasoning**",
            "",
            result["reasoning"],
            "",
            "**Output**",
            "",
            # 4 backticks: the output itself often contains ``` code fences
            "````",
            result["output"].strip(),
            "````",
        ]

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

    print(f"Wrote {path}")
    return path

In [11]:
# [11] RUN every demo iteration  --  evaluates the selected version
runs = globals().get("runs", {})

with open("dataset-aws.json", "r") as f:
    dataset = json.load(f)

print(f"Prompt version: {PROMPT_VERSION}")
results = run_eval(dataset)
runs[PROMPT_VERSION] = results
write_results_md(results)

Prompt version: v1
Average score: 8.166666666666666
Wrote prompt-eval-results-v1.md


'prompt-eval-results-v1.md'

In [12]:
# [12] Raw graded results (optional)
print(json.dumps(results, indent=2))

[
  {
    "output": "\n^(?P<timestamp>\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}\\.\\d{3}Z)\\s+(?P<level>\\[?(?:DEBUG|INFO|WARN|WARNING|ERROR|FATAL|CRITICAL)\\]?)\\s+(?P<message>.*)$\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message using a regular expression",
      "format": "regex",
      "solution_criteria": "The regex should correctly capture timestamp in ISO 8601 format, log level (INFO, ERROR, WARN, DEBUG), and the remaining message text from a CloudWatch log line"
    },
    "score": 7.5,
    "model_score": 5,
    "syntax_score": 10,
    "reasoning": "The regex demonstrates good understanding of the core requirements with proper named groups and ISO 8601 timestamp matching. However, it has a critical syntax error in the bracket handling that would cause failures on bracketed log levels. Additionally, it lacks flexibility for case variations and multiline content that are realistic in CloudWatch scenarios. Th

In [13]:
# [13] RUN every demo iteration  --  compares all versions so far
def compare_runs(runs):
    if not runs:
        print("No runs yet.")
        return

    versions = list(runs)
    tasks = [result["test_case"]["task"] for result in next(iter(runs.values()))]

    header = "| # | Format | " + " | ".join(versions) + " |"
    divider = "| --- | --- | " + " | ".join("---" for _ in versions) + " |"
    lines = [header, divider]

    for index, task in enumerate(tasks):
        format = runs[versions[0]][index]["test_case"]["format"]
        scores = [f"{runs[version][index]['score']}" for version in versions]
        lines.append(f"| {index + 1} | {format} | " + " | ".join(scores) + " |")

    averages = [f"**{mean([r['score'] for r in runs[v]]):.2f}**" for v in versions]
    lines.append("| | **average** | " + " | ".join(averages) + " |")

    print("\n".join(lines))

    for index, task in enumerate(tasks):
        print(f"\n{index + 1}. {task}")
        for version in versions:
            result = runs[version][index]
            print(
                f"   {version}: {result['score']}"
                f" (syntax {result['syntax_score']}, model {result['model_score']})"
            )


compare_runs(runs)

| # | Format | v1 |
| --- | --- | --- |
| 1 | regex | 7.5 |
| 2 | python | 8.5 |
| 3 | json | 8.5 |
| | **average** | **8.17** |

1. Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message using a regular expression
   v1: 7.5 (syntax 10, model 5)

2. Write a Python function that takes an AWS S3 bucket name and returns True if it follows AWS naming conventions (lowercase, 3-63 characters, no consecutive hyphens)
   v1: 8.5 (syntax 10, model 7)

3. Create a JSON configuration object for an AWS Lambda function that specifies runtime as Python 3.11, memory as 512 MB, timeout as 60 seconds, and includes environment variables for API_KEY and REGION
   v1: 8.5 (syntax 10, model 7)


In [ ]:
# [14] DEMO FLOW  
#
# Cell numbers below are the stable [N] markers in each cell's first line.
# They are NOT the [n] execution counters VS Code shows on the left - those
# re-label on every run. Count by the marker, not by the gutter.
#
# ---------------------------------------------------------------------------
# ONE-TIME SETUP (before the audience arrives)
# ---------------------------------------------------------------------------
#   Run [1] -> [13], SKIPPING [4].
#   That establishes the v1 baseline and writes prompt-eval-results-v1.md.
#   Confirm dataset-aws.json is committed so it cannot drift mid-demo.
#
# ---------------------------------------------------------------------------
# EACH ITERATION (v1 -> v2 -> v3 ...)
# ---------------------------------------------------------------------------
#   1. Open prompt-eval-results-<version>.md and read the Weaknesses aloud.
#      That is the evidence for what you are about to change.
#   2. In [6]: add the next prompt_vN function, register it in PROMPTS,
#      and set PROMPT_VERSION = "vN".  RUN [6].
#   3. RUN [11]  -> evaluates the new version, writes prompt-eval-results-vN.md,
#                   and stores the run in `runs` for comparison.
#   4. RUN [13]  -> side-by-side table of every version so far.
#   ([12] is optional - shows the raw graded JSON.)
#
# ---------------------------------------------------------------------------
# RULES THAT KEEP THE NUMBERS HONEST
# ---------------------------------------------------------------------------
#   * Dont run [4]. It regenerates dataset-aws.json and voids every
#     previous score - you would be comparing across different test sets.
#   * NEVER restart the kernel mid-demo. `runs` lives in memory, so a restart
#     loses the earlier versions and [13] will only show the current one.
#     (The per-version .md files survive; the comparison table does not.)
#   * temperature=0 in [2] so score changes come from the prompt, not sampling.
#
# ---------------------------------------------------------------------------
# WHAT TO POINT AT
# ---------------------------------------------------------------------------
#   Each score is (model_score + syntax_score) / 2, and [13] prints the split.
#   Syntax is binary 0 or 10 - it only catches wrong-format answers.
#   Model score is the graded rubric and is where most improvement shows up.
#   Naming the format fixes syntax; showing the criteria and cutting scope
#   creep moves the model score.
